# MNIST MLP3 — Muon + auxiliary AdamW baseline

This is a clean three-seed optimizer control with no RG intervention. The committed profile uses optimizer-specific linear warm-up followed by cosine decay to a non-zero learning-rate floor.

Reference profile: Muon on fc1/fc2 at LR 0.02 with momentum 0.95 and five Newton–Schulz steps; fc3 and biases use auxiliary AdamW at LR 3e-4. Both use two-epoch warm-up and cosine decay.

At epoch zero and after every epoch, WeightWatcher runs with `ERG=True, randomize=True`; direct `alpha`, `ERG_gap`, and `num_traps` outputs are retained without proxy values.


In [ ]:
from pathlib import Path
import os, sys

ROOT = None
for path in [Path.cwd(), *Path.cwd().parents]:
    candidate = path / 'baseline'
    if (candidate / 'rg_baselines').is_dir():
        ROOT = candidate
        break
    if (path / 'rg_baselines').is_dir():
        ROOT = path
        break
if ROOT is None:
    raise RuntimeError('Run this notebook from a clone of CalculatedContent/rg_optimizers.')
ROOT = ROOT.resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

def resolve_dir(name, default):
    raw = os.environ.get(name)
    path = Path(raw).expanduser() if raw else default
    return (path if path.is_absolute() else Path.cwd() / path).resolve()

RUN_ROOT = resolve_dir('RG_BASELINE_RUN_ROOT', ROOT / 'runs')
DATA_DIR = resolve_dir('RG_BASELINE_DATA_DIR', ROOT / 'data')
RUN_ROOT.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
print('run root:', RUN_ROOT)
print('data root:', DATA_DIR)


In [ ]:
from dataclasses import asdict
from IPython.display import display
import pandas as pd
import matplotlib.pyplot as plt

from rg_baselines import (
    BaselineConfig,
    DEFAULT_BASELINE_SEEDS,
    plot_all_replicates,
    run_baseline_replicates,
)

pd.set_option('display.max_columns', None)
SEEDS = DEFAULT_BASELINE_SEEDS
CONFIG = BaselineConfig(optimizer='sgd_momentum_muon', save_epoch_checkpoints=True)
RUN_DIR = RUN_ROOT / CONFIG.run_slug
PLOT_DIR = RUN_DIR / 'plots'
display(pd.DataFrame([asdict(CONFIG)]))
print('seeds:', SEEDS)


In [ ]:
suite = run_baseline_replicates(
    CONFIG,
    seeds=SEEDS,
    data_dir=DATA_DIR,
    output_dir=RUN_DIR,
    progress=True,
    confidence=0.95,
)
plot_all_replicates(suite, output_dir=PLOT_DIR, show=True)
print('saved:', RUN_DIR)


## Per-epoch performance and learning-rate schedule


In [ ]:
display(
    suite.performance[
        ['seed', 'epoch', 'primary_lr', 'auxiliary_lr',
         'train_loss', 'test_loss', 'train_accuracy', 'test_accuracy']
    ].sort_values(['seed', 'epoch'])
)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for metric, linestyle in (('primary_lr', '-'), ('auxiliary_lr', '--')):
    values = suite.performance.groupby('epoch')[metric].mean()
    if values.notna().any():
        ax.plot(values.index, values.values, linestyle=linestyle, label=metric)
ax.set_yscale('log')
ax.set_xlabel('Epoch')
ax.set_ylabel('Learning rate')
ax.set_title(f'{CONFIG.optimizer_label}: warm-up and cosine decay')
ax.grid(alpha=0.25)
ax.legend(frameon=False)
fig.tight_layout()
plt.show()


## WeightWatcher layer metrics

The table below contains the direct per-layer values for every seed and epoch.


In [ ]:
display(
    suite.spectral_metrics[
        ['seed', 'epoch', 'layer', 'alpha', 'ERG_gap', 'num_traps',
         'detX_num', 'num_pl_spikes', 'm_midpoint',
         'trace_log_midpoint_per_eval']
    ].sort_values(['epoch', 'layer', 'seed'])
)


## Persistence audit

Every seed must contain the final state and all epoch checkpoints.


In [ ]:
required = [
    RUN_DIR / 'performance_by_epoch_and_seed.csv',
    RUN_DIR / 'spectral_metrics_by_epoch_layer_and_seed.csv',
    RUN_DIR / 'performance_summary_95ci.csv',
    RUN_DIR / 'spectral_summary_95ci.csv',
    RUN_DIR / 'replicate_manifest.json',
]
for seed in SEEDS:
    seed_dir = RUN_DIR / 'seeds' / f'seed_{seed}'
    required.append(seed_dir / 'final_state.pt')
    required.extend(seed_dir / 'checkpoints' / f'epoch_{epoch:03d}.pt'
                    for epoch in range(1, CONFIG.epochs + 1))
missing = [path for path in required if not path.is_file()]
if missing:
    raise RuntimeError('missing persisted artifacts:\n' + '\n'.join(map(str, missing)))
print('verified files:', len(required))
